# Checkpoint 4
## RM566515 - Arthur dos Santos Cabral

# 1. Instalando SDK

In [1]:
!pip install -q -U "google-genai>=2.3.0" "pydantic>=2.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.6/472.6 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.7/259.7 kB 7.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.57.0 which is incompatible.


## 2. Crie o cliente

In [38]:
from google import genai
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

MODEL = "gemini-3.7-flash"
print("Cliente criado com sucesso.")

Cliente criado com sucesso.


### 3. Definição da ficha de atendimento

In [21]:
from typing import Literal

from pydantic import BaseModel, Field


class FichaAtendimento(BaseModel):

    nome_pet: str = Field(
        description="Nome do pet"
    )

    especie: Literal["cao", "gato"] = Field(
        description="Espécie do pet"
    )

    porte: Literal["pequeno", "medio", "grande"] = Field(
        description="Porte do pet"
    )

    servicos: list[str] = Field(
        description="Lista de serviços estéticos solicitados"
    )

    dia: str = Field(
        description="Dia da semana escolhido para o atendimento"
    )

    periodo: str = Field(
        description="Período escolhido para o atendimento, manhã ou tarde"
    )

    valor_total: float = Field(
        description="Valor total do orçamento em reais"
    )


print(FichaAtendimento.model_json_schema())

{'properties': {'nome_pet': {'description': 'Nome do pet', 'title': 'Nome Pet', 'type': 'string'}, 'especie': {'description': 'Espécie do pet', 'enum': ['cao', 'gato'], 'title': 'Especie', 'type': 'string'}, 'porte': {'description': 'Porte do pet', 'enum': ['pequeno', 'medio', 'grande'], 'title': 'Porte', 'type': 'string'}, 'servicos': {'description': 'Lista de serviços estéticos solicitados', 'items': {'type': 'string'}, 'title': 'Servicos', 'type': 'array'}, 'dia': {'description': 'Dia da semana escolhido para o atendimento', 'title': 'Dia', 'type': 'string'}, 'periodo': {'description': 'Período escolhido para o atendimento, manhã ou tarde', 'title': 'Periodo', 'type': 'string'}, 'valor_total': {'description': 'Valor total do orçamento em reais', 'title': 'Valor Total', 'type': 'number'}}, 'required': ['nome_pet', 'especie', 'porte', 'servicos', 'dia', 'periodo', 'valor_total'], 'title': 'FichaAtendimento', 'type': 'object'}


## 4. Regras de negócio e controle da agenda

In [34]:
# Preços dos serviços
PRECOS = {
    "banho": {
        "pequeno": 45,
        "medio": 60,
        "grande": 80,
    },
    "tosa": {
        "pequeno": 35,
        "medio": 50,
        "grande": 65,
    },
    "hidratacao": {
        "pequeno": 20,
        "medio": 30,
        "grande": 40,
    },
}


# Agenda inicial da PetTech
AGENDA = {
    "segunda": {
        "manha": True,
        "tarde": False,
    },
    "quarta": {
        "manha": False,
        "tarde": True,
    },
    "sabado": {
        "manha": True,
        "tarde": False,
    },
}


# Lista dos agendamentos realizados
AGENDAMENTOS = []


def calcular_orcamento(porte: str, servicos: list[str]) -> dict:
    if porte not in PRECOS["banho"]:
        return {
            "erro": "Porte inválido."
        }

    total = 0

    for servico in servicos:

        if servico not in PRECOS:
            return {
                "erro": f"Serviço inválido: {servico}"
            }

        total += PRECOS[servico][porte]

    return {
        "valor_total": total
    }


def verificar_disponibilidade(dia: str, periodo: str) -> dict:
    disponivel = AGENDA[dia][periodo]

    return {
        "dia": dia,
        "periodo": periodo,
        "disponivel": disponivel
    }


def registrar_agendamento(
    nome_pet: str,
    especie: str,
    porte: str,
    servicos: list[str],
    dia: str,
    periodo: str,
    valor_total: float
) -> dict:

    # Segurança: verifica novamente antes de reservar
    if not AGENDA[dia][periodo]:
        return {
            "sucesso": False,
            "erro": "O horário não está mais disponível."
        }

    # Marca o horário como ocupado
    AGENDA[dia][periodo] = False

    agendamento = {
        "nome_pet": nome_pet,
        "especie": especie,
        "porte": porte,
        "servicos": servicos,
        "dia": dia,
        "periodo": periodo,
        "valor_total": valor_total
    }

    AGENDAMENTOS.append(agendamento)

    return {
        "sucesso": True,
        "agendamento": agendamento
    }

## 5. Declaração das ferramentas

In [23]:
FERRAMENTAS = [

    {
        "type": "function",
        "name": "calcular_orcamento",
        "description": (
            "Calcula o valor total dos serviços estéticos da PetTech "
            "com base no porte do pet e nos serviços solicitados. "
            "Nunca invente ou calcule o valor manualmente."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "porte": {
                    "type": "string",
                    "enum": ["pequeno", "medio", "grande"],
                    "description": "Porte do pet"
                },
                "servicos": {
                    "type": "array",
                    "items": {
                        "type": "string",
                        "enum": ["banho", "tosa", "hidratacao"]
                    },
                    "description": "Serviços estéticos solicitados"
                },
            },
            "required": ["porte", "servicos"],
        },
    },

    {
        "type": "function",
        "name": "verificar_disponibilidade",
        "description": (
            "Verifica se existe disponibilidade para atendimento "
            "em determinado dia da semana e período. "
            "Nunca invente ou presuma disponibilidade."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "dia": {
                    "type": "string",
                    "enum": ["segunda", "quarta", "sabado"],
                    "description": "Dia da semana desejado"
                },
                "periodo": {
                    "type": "string",
                    "enum": ["manha", "tarde"],
                    "description": "Período desejado"
                },
            },
            "required": ["dia", "periodo"],
        },
    },

]

print("Ferramentas declaradas:", [item["name"] for item in FERRAMENTAS])

Ferramentas declaradas: ['calcular_orcamento', 'verificar_disponibilidade']


## 6. Definição do comportamento do assistente

In [35]:
SYSTEM_PROMPT = "\n".join([

    "Você é a assistente virtual da PetTech, uma pet shop especializada em serviços estéticos para cães e gatos.",

    "Seu papel é atender os clientes de forma simpática, acolhedora, natural e objetiva, ajudando-os a realizar um pré-agendamento.",

    "Seja cordial e demonstre atenção ao pet do cliente, sem exagerar ou tornar a conversa artificial.",

    "A PetTech oferece somente três serviços estéticos: banho, tosa e hidratação.",

    "Use somente as ferramentas disponibilizadas pela aplicação para calcular preços e verificar disponibilidade.",

    "Nunca calcule preços por conta própria e nunca invente ou presuma horários disponíveis.",

    "Para calcular o orçamento, é necessário conhecer o porte do pet e os serviços desejados.",

    "Para verificar a disponibilidade, é necessário conhecer o dia e o período desejados.",

    "Quando faltar uma informação necessária, faça uma pergunta objetiva e simpática.",

    "Lembre-se das informações que o cliente já forneceu durante a conversa e não peça novamente algo que já foi informado.",

    "Se o cliente escolher um horário indisponível, informe isso com cordialidade e ofereça outras opções disponíveis.",

    "Antes de finalizar um agendamento, apresente um resumo com nome, espécie, porte, serviços, dia, período e valor total.",

    "Peça uma confirmação explícita do cliente antes de considerar o agendamento confirmado.",

    "Nunca considere um agendamento confirmado apenas porque o cliente demonstrou interesse.",

    "Depois que o cliente confirmar o agendamento, a aplicação deve registrar a reserva.",

    "Um horário já reservado deve ser considerado indisponível para novos agendamentos.",

    "Se o cliente informar que o pet está doente, machucado, apático ou apresentar qualquer sintoma, interrompa o atendimento estético.",

    "Nessas situações, não faça diagnóstico, não calcule orçamento, não verifique disponibilidade e não realize agendamento.",

    "Nessas situações, recomende que o cliente procure um médico-veterinário.",

    "A ficha final só deve ser criada depois que todos os dados estiverem completos, o horário tiver sido verificado, o orçamento tiver sido calculado e o cliente tiver confirmado explicitamente o agendamento.",

    "Responda sempre em português.",

])

## 7. Controle a execução no código

In [25]:
FUNCOES_AUTORIZADAS = {
    "calcular_orcamento": calcular_orcamento,
    "verificar_disponibilidade": verificar_disponibilidade,
}


def executar_ferramenta(nome: str, argumentos: dict) -> dict:

    funcao = FUNCOES_AUTORIZADAS.get(nome)

    if funcao is None:
        return {
            "erro": "ferramenta não autorizada"
        }

    try:
        return funcao(**argumentos)

    except (TypeError, KeyError) as erro:
        return {
            "erro": "argumentos inválidos",
            "detalhe": str(erro)
        }

## 8. Retorno dos resultados das ferramentas ao modelo

In [26]:
import json

def montar_resultado_ferramenta(chamada, resultado: dict) -> dict:
    return {
        "type": "function_result",
        "name": chamada.name,
        "call_id": chamada.id,
        "result": [
            {
                "type": "text",
                "text": json.dumps(resultado, ensure_ascii=False)
            }
        ],
    }

## 9. Ciclo de Function Calling

In [27]:
def atender_com_ferramentas(
    mensagem: str,
    mostrar_log: bool = True
) -> str:

    primeira_interaction = client.interactions.create(
        model=MODEL,
        system_instruction=SYSTEM_PROMPT,
        input=mensagem,
        tools=FERRAMENTAS,
    )


    chamadas = [
        etapa
        for etapa in primeira_interaction.steps
        if etapa.type == "function_call"
    ]


    if not chamadas:

        if mostrar_log:
            print("[log] Nenhuma ferramenta executada.")

        return primeira_interaction.output_text


    function_results = []


    for chamada in chamadas:

        resultado = executar_ferramenta(
            chamada.name,
            chamada.arguments
        )


        if mostrar_log:
            print(
                f"[log] {chamada.name}"
                f"({chamada.arguments})"
                f" -> {resultado}"
            )


        function_results.append(
    montar_resultado_ferramenta(chamada, resultado)
)


    interaction_final = client.interactions.create(
        model=MODEL,
        system_instruction=SYSTEM_PROMPT,
        input=function_results,
        tools=FERRAMENTAS,
        previous_interaction_id=primeira_interaction.id,
    )


    return interaction_final.output_text

## 10. Validação das regras e segurança das ferramentas

In [36]:
print(
    executar_ferramenta(
        "calcular_orcamento",
        {
            "porte": "pequeno",
            "servicos": ["banho", "tosa"]
        }
    )
)
print(
    executar_ferramenta(
        "verificar_disponibilidade",
        {
            "dia": "segunda",
            "periodo": "manha"
        }
    )
)
print(
    executar_ferramenta(
        "apagar_dados_pettech",
        {}
    )
)
print(
    executar_ferramenta(
        "calcular_orcamento",
        {
            "porte": "gigante",
            "servicos": ["banho"]
        }
    )
)

{'valor_total': 80}
{'dia': 'segunda', 'periodo': 'manha', 'disponivel': True}
{'erro': 'ferramenta não autorizada'}
{'erro': 'Porte inválido.'}


# 11. Memória da conversa e controle dos agendamentos

In [29]:
class SessaoPetTech:
    def __init__(self):
        self.interaction_id = None
        self.aguardando_confirmacao = False
        self.ultimo_orcamento = None
        self.ultimo_horario = None
        self.bloqueado_veterinario = False


def nova_sessao():
    return SessaoPetTech()

def eh_confirmacao(mensagem: str) -> bool:
    texto = mensagem.lower().strip()

    confirmacoes = [
        "sim",
        "sim!",
        "pode",
        "pode sim",
        "confirmo",
        "confirmado",
        "pode confirmar",
        "pode agendar",
        "pode marcar",
        "pode realizar",
        "fechado",
        "está confirmado",
        "tudo certo",
        "pode fechar",
    ]

    return any(
        texto == frase or texto.startswith(frase + " ")
        for frase in confirmacoes
    )

def resposta_pede_confirmacao(resposta: str) -> bool:
    texto = resposta.lower()

    termos = [
        "posso confirmar",
        "confirma o agendamento",
        "deseja confirmar",
        "posso realizar o agendamento",
        "posso agendar",
        "podemos confirmar",
        "confirma para mim",
    ]

    return any(termo in texto for termo in termos)


def resetar_agenda():
    AGENDA["segunda"]["manha"] = True
    AGENDA["segunda"]["tarde"] = False

    AGENDA["quarta"]["manha"] = False
    AGENDA["quarta"]["tarde"] = True

    AGENDA["sabado"]["manha"] = True
    AGENDA["sabado"]["tarde"] = False

    AGENDAMENTOS.clear()

    print("Agenda restaurada para o estado inicial.")


def registrar_agendamento(
    nome_pet: str,
    especie: str,
    porte: str,
    servicos: list[str],
    dia: str,
    periodo: str
) -> dict:

    # Confirma novamente a disponibilidade antes de reservar.
    # Isso impede duas reservas no mesmo horário.
    disponibilidade = verificar_disponibilidade(dia, periodo)

    if not disponibilidade["disponivel"]:
        return {
            "sucesso": False,
            "erro": "O horário não está mais disponível."
        }

    # Calcula novamente o preço usando a regra oficial.
    # Nunca confiamos em um valor digitado pelo modelo.
    orcamento = calcular_orcamento(porte, servicos)

    if "erro" in orcamento:
        return {
            "sucesso": False,
            "erro": orcamento["erro"]
        }

    valor_total = orcamento["valor_total"]

    agendamento = {
        "nome_pet": nome_pet,
        "especie": especie,
        "porte": porte,
        "servicos": servicos,
        "dia": dia,
        "periodo": periodo,
        "valor_total": valor_total,
    }

    # Marca o horário como ocupado.
    AGENDA[dia][periodo] = False

    # Guarda o agendamento realizado.
    AGENDAMENTOS.append(agendamento)

    return {
        "sucesso": True,
        "agendamento": agendamento
    }

# 12. Atendimento completo da PetTech

In [30]:
def atender_sessao(
    sessao: SessaoPetTech,
    mensagem: str,
    mostrar_log: bool = True
) -> str:

    # ---------------------------------------------------------
    # 1. Se o atendimento já foi bloqueado por motivo veterinário
    # ---------------------------------------------------------

    if sessao.bloqueado_veterinario:
        return (
            "Como você relatou sintomas no seu pet, o atendimento "
            "estético permanece interrompido. Recomendo procurar "
            "um médico-veterinário antes de realizar qualquer serviço."
        )

    # ---------------------------------------------------------
    # 2. Verifica se o cliente está confirmando um agendamento
    # ---------------------------------------------------------

    if (
        sessao.aguardando_confirmacao
        and eh_confirmacao(mensagem)
    ):

        if mostrar_log:
            print("[log] Cliente confirmou o agendamento.")

        # Pedimos ao Gemini a ficha estruturada utilizando
        # o histórico da conversa.
        interaction_ficha = client.interactions.create(
            model=MODEL,
            system_instruction=(
                SYSTEM_PROMPT
                + "\n\n"
                "O cliente confirmou explicitamente o agendamento. "
                "Agora gere somente a ficha final estruturada. "
                "Utilize exclusivamente as informações presentes "
                "no histórico da conversa e os valores retornados "
                "pelas ferramentas."
            ),
            input=(
                "O cliente confirmou explicitamente o agendamento. "
                "Gere a ficha final."
            ),
            previous_interaction_id=sessao.interaction_id,
            response_format={
                "type": "text",
                "mime_type": "application/json",
                "schema": FichaAtendimento.model_json_schema(),
            },
        )

        # Validação rigorosa com Pydantic.
        ficha = FichaAtendimento.model_validate_json(
            interaction_ficha.output_text
        )

        if mostrar_log:
            print("[log] Ficha Pydantic validada com sucesso.")

        # -----------------------------------------------------
        # 3. Registra efetivamente o agendamento
        # -----------------------------------------------------

        resultado_agendamento = registrar_agendamento(
            nome_pet=ficha.nome_pet,
            especie=ficha.especie,
            porte=ficha.porte,
            servicos=ficha.servicos,
            dia=ficha.dia,
            periodo=ficha.periodo,
        )

        if not resultado_agendamento["sucesso"]:

            sessao.aguardando_confirmacao = False

            return (
                "Poxa, esse horário acabou de ficar indisponível. "
                "Não consegui confirmar o agendamento. "
                "Posso verificar outro horário para você."
            )

        # -----------------------------------------------------
        # 4. Agendamento confirmado
        # -----------------------------------------------------

        sessao.interaction_id = interaction_ficha.id
        sessao.aguardando_confirmacao = False

        # Garante que o valor apresentado na ficha corresponde
        # ao valor calculado pela regra de negócio.
        ficha.valor_total = resultado_agendamento[
            "agendamento"
        ]["valor_total"]

        return (
            "Perfeito! O agendamento foi confirmado. "
            "Aqui está a ficha de atendimento:\n\n"
            + ficha.model_dump_json(
                indent=2,
                ensure_ascii=False
            )
        )

    # ---------------------------------------------------------
    # 5. Envia a mensagem para o Gemini
    # ---------------------------------------------------------

    argumentos_interacao = {
        "model": MODEL,
        "system_instruction": SYSTEM_PROMPT,
        "input": mensagem,
        "tools": FERRAMENTAS,
    }

    # Se já existe histórico, continua a mesma conversa.
    if sessao.interaction_id is not None:
        argumentos_interacao["previous_interaction_id"] = (
            sessao.interaction_id
        )

    interaction = client.interactions.create(
        **argumentos_interacao
    )

    # ---------------------------------------------------------
    # 6. Verifica se o Gemini solicitou ferramentas
    # ---------------------------------------------------------

    chamadas = [
        etapa
        for etapa in interaction.steps
        if etapa.type == "function_call"
    ]

    if not chamadas:

        resposta = interaction.output_text

        sessao.interaction_id = interaction.id

        # Detecta se o modelo está pedindo confirmação.
        if resposta_pede_confirmacao(resposta):
            sessao.aguardando_confirmacao = True

        # Se o modelo indicar uma situação veterinária,
        # encerramos o atendimento estético.
        termos_veterinarios = [
            "médico-veterinário",
            "veterinário",
            "veterinário(a)",
        ]

        sintomas = [
            "doente",
            "machucado",
            "apático",
            "vomitando",
            "vômito",
            "olho vermelho",
            "sintoma",
        ]

        if (
            any(t in mensagem.lower() for t in sintomas)
            and any(t in resposta.lower() for t in termos_veterinarios)
        ):
            sessao.bloqueado_veterinario = True
            sessao.aguardando_confirmacao = False

        return resposta

    # ---------------------------------------------------------
    # 7. Executa as ferramentas solicitadas pelo Gemini
    # ---------------------------------------------------------

    function_results = []

    for chamada in chamadas:

        resultado = executar_ferramenta(
            chamada.name,
            chamada.arguments
        )

        if mostrar_log:
            print(
                f"[log] {chamada.name}"
                f"({chamada.arguments}) -> {resultado}"
            )

        # Guarda informações úteis da última consulta.
        if chamada.name == "calcular_orcamento":
            sessao.ultimo_orcamento = resultado

        elif chamada.name == "verificar_disponibilidade":
            sessao.ultimo_horario = resultado

        function_results.append(
            montar_resultado_ferramenta(
                chamada,
                resultado
            )
        )

    # ---------------------------------------------------------
    # 8. Devolve os resultados ao Gemini mantendo o histórico
    # ---------------------------------------------------------

    interaction_final = client.interactions.create(
        model=MODEL,
        system_instruction=SYSTEM_PROMPT,
        input=function_results,
        tools=FERRAMENTAS,
        previous_interaction_id=interaction.id,
    )

    sessao.interaction_id = interaction_final.id

    resposta = interaction_final.output_text

    # Verifica se agora o modelo está esperando confirmação.
    if resposta_pede_confirmacao(resposta):
        sessao.aguardando_confirmacao = True

    return resposta

In [31]:
resetar_agenda()

Agenda restaurada para o estado inicial.


# 13. Simulações

In [ ]:
# ============================================================
# 13. SIMULAÇÕES OBRIGATÓRIAS
# ============================================================

def executar_simulacao(titulo: str, mensagens: list[str]):
    print("\n" + "=" * 70)
    print(titulo)
    print("=" * 70)

    sessao = nova_sessao()

    for numero, mensagem in enumerate(mensagens, start=1):

        print(f"\nCLIENTE:")
        print(mensagem)

        resposta = atender_sessao(
            sessao,
            mensagem,
            mostrar_log=True
        )

        print("\nPETTECH:")
        print(resposta)

    print("\n" + "-" * 70)
    print("ESTADO FINAL DA SIMULAÇÃO")
    print("-" * 70)

    print("Agendamentos registrados:")
    print(AGENDAMENTOS)

    print("\nAgenda atual:")
    print(AGENDA)


# ============================================================
# SIMULAÇÃO 1 — CAMINHO FELIZ
# ============================================================

resetar_agenda()

executar_simulacao(
    "SIMULAÇÃO 1 — O CAMINHO FELIZ",
    [
        "Olá! Quero marcar um banho e uma tosa para o meu cachorro Thor, "
        "que é pequeno. Pode ser na segunda-feira de manhã.",

        "Sim, pode confirmar o agendamento."
    ]
)


# ============================================================
# SIMULAÇÃO 2 — CLIENTE ESQUECIDO + HORÁRIO INDISPONÍVEL
# ============================================================

resetar_agenda()

executar_simulacao(
    "SIMULAÇÃO 2 — CLIENTE ESQUECIDO E HORÁRIO LOTADO",
    [
        "Oi! Quero marcar um banho para o Rex na segunda-feira à tarde.",

        "Ah, esqueci de falar. Ele é um cachorro de porte médio.",

        "Pode ser quarta-feira à tarde então.",

        "Sim, pode confirmar."
    ]
)


# ============================================================
# SIMULAÇÃO 3 — GUARDRAIL VETERINÁRIO
# ============================================================

resetar_agenda()

executar_simulacao(
    "SIMULAÇÃO 3 — GUARDRAIL VETERINÁRIO",
    [
        "Quero marcar um banho pro meu gato grande, mas ele está "
        "vomitando desde ontem e o olho tá vermelho. Pode ser sábado?"
    ]
)

Agenda restaurada para o estado inicial.

SIMULAÇÃO 1 — O CAMINHO FELIZ

CLIENTE:
Olá! Quero marcar um banho e uma tosa para o meu cachorro Thor, que é pequeno. Pode ser na segunda-feira de manhã.
